# Tigers & Goats: Latent Value Shaping(LVS) VAE Dataset Generator

## Project

**Tigers & Goats**

## Module

**LVS VAE Dataset Generator**

## Purpose / Goal

Collect balanced offline state-value data for training the **Latent Value Shaping VAE**.

## Overview

This notebook will:

- Load multiple goat policies/checkpoints
- Play full games against varied tiger opponents
- Record useful state features at every move
- Label each state using the final episode outcome
- Balance the dataset across goat wins, tiger wins, and timeouts
- Save arrays and metadata for offline VAE training

## Dataset Target

The goal is to build a dataset with approximately:

- **40%** goat-win states
- **40%** tiger-win states
- **20%** timeout / draw-like states

## Value Labels

Each state receives a value label based on the final game outcome:

| Outcome | Label |
|---|---:|
| Goat win | `1.0` |
| Tiger win | `0.0` |
| Timeout / draw | `0.5` |

## Recorded Per State

Each saved state should include:

- Raw environment state
- Board layout
- Move index
- Total moves
- Progress ratio
- Goats eaten
- Goats placed
- Phase
- Final winner
- Source policy / opponent setup

## Output

The dataset should be saved under:

```text
VAE_DATA/


---
# Implementation

### 1. Imports and Paths

In [ ]:
import sys
import time
import json
from pathlib import Path
import numpy as np
import pandas as pd

from sb3_contrib import MaskablePPO

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from env_tng_abc import (
    TnGEnv,
    TIGER_AI_GREEDY,
    TIGER_AI_SMART,
    GOAT_LEARNER
)

DATASET_ROOT = PROJECT_ROOT / "VAE_DATA"
DATASET_NAME = "end06_20k_40_40_20"
DATASET_GOAL = "endgame"  # "full" or "endgame"
ENDGAME_PROGRESS_THRESHOLD = 0.7 # Default is 0.7
VALID_DATASET_GOALS = {"full", "endgame"}

if DATASET_GOAL not in VALID_DATASET_GOALS:
    raise ValueError(f"DATASET_GOAL must be one of {VALID_DATASET_GOALS}, got {DATASET_GOAL}")

DATASET_DIR = DATASET_ROOT / DATASET_NAME
DATASET_DIR.mkdir(parents=True, exist_ok=True)

STATES_PATH = DATASET_DIR / "states_labels.npz"
DATASET_PATH = DATASET_DIR / "dataset.parquet"
METADATA_PATH = DATASET_DIR / "metadata.parquet"
DATASET_SUMMARY_PATH = DATASET_DIR / "dataset_summary.json"

### 2. Dataset Configuration

In [141]:
TARGET_TOTAL_ROWS = 20_000
MAX_EPISODES = 20_000
MAX_TURNS = 100
SEED = 42
GOAT_WIN_RATIO = 0.4
TIGER_WIN_RATIO = 0.4
TIMEOUT_RATIO = 0.2
if not np.isclose(GOAT_WIN_RATIO + TIGER_WIN_RATIO + TIMEOUT_RATIO, 1.0):
    raise ValueError("goat, tiger, and timeout ratios must sum to 1")

TARGET_ROWS = {
    "goat_win":int(TARGET_TOTAL_ROWS * GOAT_WIN_RATIO),
    "tiger_win": int(TARGET_TOTAL_ROWS * TIGER_WIN_RATIO),
    "timeout": int(TARGET_TOTAL_ROWS * TIMEOUT_RATIO), 
}

if sum(TARGET_ROWS.values()) != TARGET_TOTAL_ROWS:
    raise ValueError(f"TARGET_ROWS sum to {sum(TARGET_ROWS.values())}, expected {TARGET_TOTAL_ROWS}")

TARGET_ROWS

{'goat_win': 8000, 'tiger_win': 8000, 'timeout': 4000}

### 3. Policy Helpers

In [142]:
# Random Policy
rng = np.random.default_rng(SEED)
def random_policy (obs, mask):
    legal_actions = np.flatnonzero(mask)
    return int(rng.choice(legal_actions))

# trained model policy
# use deteriministic = False for more dataset variety
def make_model_policy(model_path, device="cpu", deterministic=False):
    model = MaskablePPO.load(str(model_path), device=device)

    def policy(obs, mask):
        obs_batch = np.asarray(obs).reshape(1, -1)
        mask_batch = np.asarray(mask, dtype=bool).reshape(1, -1)

        action, _ = model.predict(
            obs_batch,
            deterministic=deterministic,
            action_masks=mask_batch,
        )

        return int(action[0])

    return policy


### 4. Source Policy List

create lists of data sources. Each source should define:
1. policy name
2. goat policy
3. tiger opponent
4. env settings

In [143]:
# define the paths to each model
# "normal_tiger" means the greedy scripted tiger opponent in env_tng_abc.
STABLE_NORMAL_GOAT_PATH = PROJECT_ROOT / "stable_models" / "models" / "Goats" / "mppo_NormalGoat030726.zip"
STABLE_SMART_GOAT_PATH = PROJECT_ROOT / "stable_models" / "models" / "Goats" / "mppo_SmartGoat030726.zip"
STABLE_ROBUST_GOAT_PATH = PROJECT_ROOT / "stable_models" / "models" / "Goats" / "mppo_RobustGoat030726.zip"

# artifacts\DataSetCreation\checkpoints\run_20260506_111352_phase_01\cp_mppo_GvNT_run_20260506_111352_2000000_steps.zip
EARLY_NORMAL_GOAT_PATH = PROJECT_ROOT / "artifacts" / "DataSetCreation" / "checkpoints" / "run_20260506_111352_phase_01" / "cp_mppo_GvNT_run_20260506_111352_2000000_steps.zip"
EARLY_SMART_GOAT_PATH = PROJECT_ROOT / "artifacts" / "baselineGoatTrainingV3" / "checkpoints" / "goat_vs_smart_tiger_50M_20260320_231909_phase_01" / "cp_mppo_GvST_goat_vs_smart_tiger_50M_20260320_231909_5000000_steps.zip"
EARLY_ROBUST_GOAT_PATH = PROJECT_ROOT / "artifacts" / "baselineGoatTrainingV3" / "checkpoints" / "robust_goat_training_20260321_070755_phase_01" / "cp_mppo_GvMixT_robust_goat_training_20260321_070755_6000000_steps.zip"


MID_NORMAL_GOAT_PATH = PROJECT_ROOT / "artifacts" / "DataSetCreation" / "checkpoints" / "run_20260506_111352_phase_01" / "cp_mppo_GvNT_run_20260506_111352_4000000_steps.zip"
MID_SMART_GOAT_PATH = PROJECT_ROOT / "artifacts" / "baselineGoatTrainingV3" / "checkpoints" / "goat_vs_smart_tiger_50M_20260320_231909_phase_01" / "cp_mppo_GvST_goat_vs_smart_tiger_50M_20260320_231909_25000000_steps.zip"
MID_ROBUST_GOAT_PATH = PROJECT_ROOT / "artifacts" / "baselineGoatTrainingV3" / "checkpoints" / "robust_goat_training_20260321_070755_phase_03" / "cp_mppo_GvMixT_robust_goat_training_20260321_070755_27034752_steps.zip"

# collect each path and assign it a source name
SOURCE_MODELS = [
    ("early_normal_goat", EARLY_NORMAL_GOAT_PATH),
    ("early_smart_goat", EARLY_SMART_GOAT_PATH),
    ("early_robust_goat", EARLY_ROBUST_GOAT_PATH),
    ("mid_normal_goat", MID_NORMAL_GOAT_PATH),
    ("mid_smart_goat", MID_SMART_GOAT_PATH),
    ("mid_robust_goat", MID_ROBUST_GOAT_PATH),
    ("stable_normal_goat", STABLE_NORMAL_GOAT_PATH),
    ("stable_smart_goat", STABLE_SMART_GOAT_PATH),
    ("stable_robust_goat", STABLE_ROBUST_GOAT_PATH),
]

# define the opponent choices
TIGER_OPPONENTS = [
    ("normal_tiger", TIGER_AI_GREEDY),
    ("smart_tiger", TIGER_AI_SMART),
]

# define a helper function to craft a dictionary for each source
def make_source(name, policy, tiger_name, tiger_ai):
    return {
        "name": f"{name}_vs_{tiger_name}",
        "policy": policy,
        "env_kwargs": {
            "learner_role": GOAT_LEARNER,
            "tiger_ai": tiger_ai,
            "max_turns": MAX_TURNS,
        },
        "source_policy": name,
        "tiger_opponent": tiger_name,
    }

# verify if each source has a valid path
missing_paths = [path for _, path in SOURCE_MODELS if not path.exists()]
# notify if there are broken or missing paths.
if missing_paths:
    raise FileNotFoundError("Missing source model paths:\n" + "\n".join(str(path) for path in missing_paths))

# create a list of dictionaries to hold each source configuration based on policy and opponent
SOURCES = []
# create the random goat policy data sources
for tiger_name, tiger_ai in TIGER_OPPONENTS:
    SOURCES.append(make_source("random_goat", random_policy, tiger_name, tiger_ai))

# Create the trained policy data sources based on the trained policy and opponent type.
for model_name, model_path in SOURCE_MODELS:
    policy = make_model_policy(model_path, device="cpu", deterministic=False)
    for tiger_name, tiger_ai in TIGER_OPPONENTS:
        SOURCES.append(make_source(model_name, policy, tiger_name, tiger_ai))

# Check the length of the sources list, and print out each dictionary name
len(SOURCES), [source["name"] for source in SOURCES]


(20,
 ['random_goat_vs_normal_tiger',
  'random_goat_vs_smart_tiger',
  'early_normal_goat_vs_normal_tiger',
  'early_normal_goat_vs_smart_tiger',
  'early_smart_goat_vs_normal_tiger',
  'early_smart_goat_vs_smart_tiger',
  'early_robust_goat_vs_normal_tiger',
  'early_robust_goat_vs_smart_tiger',
  'mid_normal_goat_vs_normal_tiger',
  'mid_normal_goat_vs_smart_tiger',
  'mid_smart_goat_vs_normal_tiger',
  'mid_smart_goat_vs_smart_tiger',
  'mid_robust_goat_vs_normal_tiger',
  'mid_robust_goat_vs_smart_tiger',
  'stable_normal_goat_vs_normal_tiger',
  'stable_normal_goat_vs_smart_tiger',
  'stable_smart_goat_vs_normal_tiger',
  'stable_smart_goat_vs_smart_tiger',
  'stable_robust_goat_vs_normal_tiger',
  'stable_robust_goat_vs_smart_tiger'])

### 5. Episode Rollout / Data Collection and Other Function Definitions

In [144]:
def outcome_bucket_and_label(winner):
    # Convert env winner string into dataset category and label value

    if winner == "Goat":
        bucket = "goat_win"
        value_label = 1.0

    elif winner == "Tiger":
        bucket = "tiger_win"
        value_label = 0.0

    else:
        # MaxTimeout, RepeatTimeout, Unknown, etc.
        bucket = "timeout"
        value_label = 0.5

    return bucket, value_label


def play_episode(source, episode_id):
    # Create env from the selected source
    env = TnGEnv(**source["env_kwargs"])

    # Start episode
    obs, info = env.reset()

    episode_states = []
    episode_actions = []
    episode_move_indices = []
    metadata_rows = []

    done = False
    move_index = 0

    while not done:
        # Get legal-action mask from env
        mask = env.get_action_mask()

        # Ask this source policy for an action
        action = source["policy"](obs, mask)

        # save current state begore taking action
        episode_states.append(obs.copy())
        episode_actions.append(action)
        episode_move_indices.append(move_index)

        # step env
        obs, reward, terminated, truncated, info = env.step(action)

        # Advance loop state
        done = terminated or truncated
        move_index += 1

    # Episode is over here 
    winner = info.get("winner", "unknown")
    bucket, value_label = outcome_bucket_and_label(winner)

    total_moves = len(episode_states)


    states = []
    labels = []
    metadata_rows = []

    # loop through each element in each list and save all data to metadata
    for state, action, move_index in zip(
        episode_states, 
        episode_actions, 
        episode_move_indices
        ):

        board = state[:23]
        goats_eaten = int(state[23])
        phase = int(state[24])
        goats_on_board = int((board == 1).sum())
        goats_placed = goats_on_board + goats_eaten

        progress_ratio = move_index/ max(total_moves, 1)

        states.append(state)
        labels.append(value_label)

        metadata_rows.append({
            "episode_id": episode_id,
            "source_name": source["name"],
            "source_policy": source["source_policy"],
            "tiger_opponent": source["tiger_opponent"],

            "move_index": move_index,
            "total_moves": total_moves,
            "progress_ratio": progress_ratio,
            "action": action,

            "goats_eaten": goats_eaten,
            "goats_on_board": goats_on_board,
            "goats_placed": goats_placed,
            "phase": phase,

            "winner": winner,
            "bucket": bucket,
            "value_label": value_label,
        })

    env.close()

    return {
        "bucket":bucket,
        "winner": winner,
        "states": states,
        "labels": labels,
        "metadata_rows": metadata_rows,
        "total_moves": total_moves,
        "source_name": source["name"]
    }

# Functions to calculate elapsed time
def start_timer():
    return time.perf_counter()

def elapsed_time(start, label="Elapsed time"):
    elapsed = time.perf_counter() - start

    hours = int(elapsed // 3600)
    minutes = int((elapsed % 3600) // 60)
    seconds = elapsed % 60

    if hours:
        formatted = f"{hours}h {minutes}m {seconds:.2f}s"
    elif minutes:
        formatted = f"{minutes}m {seconds:.2f}s"
    else:
        formatted = f"{seconds:.2f}s"

    print(f"{label}: {formatted}")
    return elapsed

### 6. Dataset Balancing Loop

In [145]:
timer_start = start_timer()

# Stores bucket information by target outcome.
bucket_states   = {bucket: [] for bucket in TARGET_ROWS}
bucket_labels   = {bucket: [] for bucket in TARGET_ROWS}
bucket_metadata = {bucket: [] for bucket in TARGET_ROWS}

episode_id = 0
source_index = 0

# this will return current dataset counts based on buckets
def bucket_counts():
    return {bucket: len(rows) for bucket, rows in bucket_states.items()}

# this will check whether each bucket has the required abount of rows
def targets_met():
    return all(bucket_counts()[bucket] >= target for bucket, target in TARGET_ROWS.items())

def select_dataset_goal_rows(result):
    if DATASET_GOAL == "full":
        return result["states"], result["labels"], result["metadata_rows"]

    selected_states = []
    selected_labels = []
    selected_metadata_rows = []

    for state, label, metadata_row in zip(
        result["states"],
        result["labels"],
        result["metadata_rows"],
    ):
        if metadata_row["progress_ratio"] >= ENDGAME_PROGRESS_THRESHOLD:
            selected_states.append(state)
            selected_labels.append(label)
            selected_metadata_rows.append(metadata_row)

    return selected_states, selected_labels, selected_metadata_rows

# main collection loop
while episode_id < MAX_EPISODES and not targets_met():
    # pick the current source, take the mod of the number of sources for wrapping index
    source = SOURCES[source_index % len(SOURCES)]
    # play a full episode based on the source, and record results
    result = play_episode(source, episode_id)

    # this will check which bucket the episode belongs to and how many rows the bucket needs
    bucket = result["bucket"]
    selected_states, selected_labels, selected_metadata_rows = select_dataset_goal_rows(result)
    rows_needed = TARGET_ROWS[bucket] - len(bucket_states[bucket])

    # This will prevent any bucket from being overfilled
    if rows_needed > 0 and selected_states:
        rows_to_take = min(rows_needed, len(selected_states))

        bucket_states[bucket].extend(selected_states[:rows_to_take])
        bucket_labels[bucket].extend(selected_labels[:rows_to_take])
        bucket_metadata[bucket].extend(selected_metadata_rows[:rows_to_take])

    episode_id += 1
    source_index += 1

    # this will print a progress report every 25 episodes
    if episode_id % 25 == 0:
        print(f"episodes={episode_id} counts={bucket_counts()}")

# create and print a final summary
final_counts = bucket_counts()
print(f"Finished after {episode_id} episodes")
print("Final counts:", final_counts)
print("Targets:", TARGET_ROWS)
if not targets_met():
    raise RuntimeError(
        f"Dataset target not met for DATASET_GOAL={DATASET_GOAL!r} after {MAX_EPISODES} episodes. "
        f"Final counts={final_counts}, targets={TARGET_ROWS}."
    )

# collect all buckets into a collective list
all_states = []
all_labels = []
all_metadata_rows = []

# 
for bucket in TARGET_ROWS:
    all_states.extend(bucket_states[bucket])
    all_labels.extend(bucket_labels[bucket])
    all_metadata_rows.extend(bucket_metadata[bucket])

# Convert the lists into final objects
states = np.asarray(all_states, dtype=np.int8)
labels = np.asarray(all_labels, dtype=np.float32)
metadata_df = pd.DataFrame(all_metadata_rows)

states.shape, labels.shape, metadata_df.shape

time_elapsed = elapsed_time(timer_start, "Dataset Generation Time")

episodes=25 counts={'goat_win': 117, 'tiger_win': 111, 'timeout': 8}
episodes=50 counts={'goat_win': 208, 'tiger_win': 235, 'timeout': 23}
episodes=75 counts={'goat_win': 395, 'tiger_win': 315, 'timeout': 23}
episodes=100 counts={'goat_win': 538, 'tiger_win': 425, 'timeout': 32}
episodes=125 counts={'goat_win': 657, 'tiger_win': 537, 'timeout': 32}
episodes=150 counts={'goat_win': 714, 'tiger_win': 694, 'timeout': 72}
episodes=175 counts={'goat_win': 885, 'tiger_win': 771, 'timeout': 84}
episodes=200 counts={'goat_win': 1105, 'tiger_win': 837, 'timeout': 95}
episodes=225 counts={'goat_win': 1193, 'tiger_win': 954, 'timeout': 95}
episodes=250 counts={'goat_win': 1313, 'tiger_win': 1066, 'timeout': 110}
episodes=275 counts={'goat_win': 1516, 'tiger_win': 1133, 'timeout': 140}
episodes=300 counts={'goat_win': 1650, 'tiger_win': 1272, 'timeout': 190}
episodes=325 counts={'goat_win': 1773, 'tiger_win': 1393, 'timeout': 190}
episodes=350 counts={'goat_win': 1916, 'tiger_win': 1478, 'timeout'

### 7. Feature and Label DataFrame

In [146]:
# create column names for the 23 board positions
board_columns = [f"cell_{idx:02d}" for idx in range(23)]
# add state column names to the board columns
state_columns = board_columns + ["goats_eaten_state", "phase_state"]

# Create a dataframe
features_df = pd.DataFrame(states, columns=state_columns)
features_df["value_label"] = labels

# Training dataset: VAE inputs plus the supervised value-head target.
# Rollout/debug fields stay in metadata_df instead of being mixed into model data.
dataset_df = features_df.copy()

# Metadata dataset: rollout/source/debug fields without columns already represented in dataset_df.
metadata_repeat_columns = ["goats_eaten", "phase", "value_label"]
metadata_dataset_df = metadata_df.drop(
    columns=metadata_repeat_columns,
    errors="ignore",
).copy()

display(metadata_dataset_df)
display(dataset_df)


,episode_id,source_name,source_policy,tiger_opponent,move_index,total_moves,progress_ratio,action,goats_on_board,goats_placed,winner,bucket
0,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,12,20,0.60,40,11,12,Goat,goat_win
1,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,13,20,0.65,25,12,13,Goat,goat_win
2,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,14,20,0.70,100,13,14,Goat,goat_win
3,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,15,20,0.75,107,14,15,Goat,goat_win
4,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,16,20,0.80,111,14,15,Goat,goat_win
...,...,...,...,...,...,...,...,...,...,...,...,...
19995,3094,stable_normal_goat_vs_normal_tiger,stable_normal_goat,normal_tiger,77,100,0.77,38,15,15,MaxTimeout,timeout
19996,3094,stable_normal_goat_vs_normal_tiger,stable_normal_goat,normal_tiger,78,100,0.78,71,15,15,MaxTimeout,timeout
19997,3094,stable_normal_goat_vs_normal_tiger,stable_normal_goat,normal_tiger,79,100,0.79,87,15,15,MaxTimeout,timeout
19998,3094,stable_normal_goat_vs_normal_tiger,stable_normal_goat,normal_tiger,80,100,0.80,64,15,15,MaxTimeout,timeout


,cell_00,cell_01,cell_02,cell_03,cell_04,cell_05,cell_06,cell_07,cell_08,cell_09,...,cell_16,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,goats_eaten_state,phase_state,value_label
0,1,1,2,2,0,0,1,2,0,1,...,1,0,0,1,0,1,0,1,0,1.0
1,1,1,2,0,2,0,1,2,1,1,...,1,0,0,1,0,1,0,1,0,1.0
2,1,1,2,2,0,1,1,2,1,1,...,1,0,0,1,0,1,0,1,0,1.0
3,1,1,2,0,2,1,1,2,1,1,...,1,0,0,1,1,1,0,1,1,1.0
4,1,1,0,2,2,1,1,2,1,1,...,1,0,0,1,1,0,1,1,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,1,1,1,2,2,0,1,1,0,1,...,1,1,0,0,1,1,1,0,1,0.5
19996,1,1,1,2,0,2,1,0,0,1,...,1,1,0,0,1,1,1,0,1,0.5
19997,1,1,1,0,2,2,1,0,1,1,...,1,1,0,0,1,1,1,0,1,0.5
19998,1,1,1,0,2,2,1,0,1,1,...,1,2,1,0,1,1,1,0,1,0.5


### 8. Save Dataset

In [147]:
# Compact arrays for model training.
state_values = dataset_df[state_columns].to_numpy(dtype=np.int8)
value_labels = dataset_df["value_label"].to_numpy(dtype=np.float32)


np.savez_compressed(
    STATES_PATH,
    states=state_values,
    labels=value_labels,
    state_columns=np.asarray(state_columns),
)

# Dataframes for inspection, filtering, and reproducibility.
def save_dataframe(df, parquet_path):
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except (ImportError, ValueError) as exc:
        csv_path = parquet_path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        print(f"Parquet unavailable for {parquet_path.name}; saved CSV instead: {csv_path}")
        return csv_path

dataset_table_path = save_dataframe(dataset_df, DATASET_PATH)
metadata_table_path = save_dataframe(metadata_dataset_df, METADATA_PATH)

if DATASET_GOAL == "endgame" and (metadata_dataset_df["progress_ratio"] < ENDGAME_PROGRESS_THRESHOLD).any():
    raise ValueError("Saved endgame dataset contains rows below the progress threshold")

dataset_summary = {
    "dataset_name": DATASET_NAME,
    "dataset_goal": DATASET_GOAL,
    "dataset_dir": str(DATASET_DIR),
    "target_total_rows": int(TARGET_TOTAL_ROWS),
    "target_rows": {key: int(value) for key, value in TARGET_ROWS.items()},
    "actual_total_rows": int(len(dataset_df)),
    "actual_bucket_counts": {
        str(key): int(value)
        for key, value in metadata_dataset_df["bucket"].value_counts().sort_index().items()
    },
    "actual_value_label_counts": {
        str(key): int(value)
        for key, value in dataset_df["value_label"].value_counts().sort_index().items()
    },
    "endgame_progress_threshold": ENDGAME_PROGRESS_THRESHOLD,
    "min_progress_ratio": float(metadata_dataset_df["progress_ratio"].min()),
    "max_progress_ratio": float(metadata_dataset_df["progress_ratio"].max()),
    "state_columns": list(state_columns),
}

with open(DATASET_SUMMARY_PATH, "w", encoding="utf-8") as handle:
    json.dump(dataset_summary, handle, indent=2)

save_summary = pd.DataFrame(
    [
        {
            "artifact": "states_labels",
            "path": str(STATES_PATH),
            "rows": len(state_values),
            "columns": state_values.shape[1],
        },
        {
            "artifact": "dataset_df",
            "path": str(dataset_table_path),
            "rows": len(dataset_df),
            "columns": dataset_df.shape[1],
        },
        {
            "artifact": "metadata_dataset_df",
            "path": str(metadata_table_path),
            "rows": len(metadata_dataset_df),
            "columns": metadata_dataset_df.shape[1],
        },
        {
            "artifact": "dataset_summary",
            "path": str(DATASET_SUMMARY_PATH),
            "rows": 1,
            "columns": len(dataset_summary),
        },
    ]
)

save_summary


Parquet unavailable for dataset.parquet; saved CSV instead: C:\Users\chris\OneDrive\Documents\GitHub\TNG_Falcon\VAE_DATA\end06_20k_40_40_20\dataset.csv
Parquet unavailable for metadata.parquet; saved CSV instead: C:\Users\chris\OneDrive\Documents\GitHub\TNG_Falcon\VAE_DATA\end06_20k_40_40_20\metadata.csv


,artifact,path,rows,columns
0,states_labels,C:\Users\chris\OneDrive\Documents\GitHub\TNG_F...,20000,25
1,dataset_df,C:\Users\chris\OneDrive\Documents\GitHub\TNG_F...,20000,26
2,metadata_dataset_df,C:\Users\chris\OneDrive\Documents\GitHub\TNG_F...,20000,12
3,dataset_summary,C:\Users\chris\OneDrive\Documents\GitHub\TNG_F...,1,12
